# gf_mul Trace Collection

ChipWhisperer로 `gf_mul(a, b)` 실행 중 power trace를 수집하여 HDF5로 저장한다.

- 펌웨어 빌드 및 업로드
- Scope 설정, gain sweep
- 랜덤 `(a, b)` 쌍에 대한 배치 캡처
- 결과 저장: `batches_gf_mul_<PLATFORM>/<N>trace_<S>point_seed<SEED>.h5`


In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F4'
SS_VER = 'SS_VER_2_1'

## 1. Imports

In [ ]:
import chipwhisperer as cw
import numpy as np
import h5py
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import time
from pathlib import Path
from datetime import datetime

## 2. Build firmware (SS v2.1)

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
echo "PLATFORM=$1"
echo "SS_VER=$2"
cd ../../firmware/mcu/hqc-sasca/
make clean PLATFORM=$1 CRYPTO_TARGET=NONE
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 V=1 2>&1 | tee build.log

## 3. Connect & Program (SS v2.1)

In [ ]:
scope = cw.scope()
scope.default_setup()

target = cw.target(scope, cw.targets.SimpleSerial2)
target.baud = 230400

prog = cw.programmers.STM32FProgrammer
fw_path = "../../firmware/mcu/hqc-sasca/hqc-gf-mul-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
print("Firmware uploaded (SS v2.1):", fw_path)

## 4. Parameters

In [ ]:
# 논문 기준: 1GHz × 1.3us ≈ 1300 samples
# 현재 환경(CW + STM32 + power)에서는 1400 ~ 2000 권장
# (1400은 gf_mul 실행 중간에 잘릴 수 있음. 2000이면 g_result store까지 포함)
SAMPLES = 1400
ADC_OFFSET = 0

# gain sweep / manual gain
REPLAY = 10
GAIN_DB = 25

# 논문은 총 500,000 traces를 수집했지만,
# 현재 환경에서는 먼저 5k ~ 10k로 leakage 확인 후 늘리는 것을 권장
N_TRACES_PREVIEW = 200
N_TRACES_CAPTURE = 100000
N_TRACES_PAPER_TARGET = 500_000

# 저장 경로
BATCH_DIR = "batches_gf_mul_{}".format(PLATFORM)
os.makedirs(BATCH_DIR, exist_ok=True)

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")

# 재현성
RNG_SEED = 12

OUT_PATH = os.path.join(BATCH_DIR, f"{N_TRACES_CAPTURE}trace_{SAMPLES}point_seed{RNG_SEED}.h5")

In [ ]:
print("target clk =", scope.clock.clkgen_freq)
print("adc freq   =", scope.clock.adc_freq)
print("ratio      =", scope.clock.adc_freq / scope.clock.clkgen_freq)

## 5. Scope / Target helpers

In [ ]:
def configure_scope(samples: int, gain_db: float):
    scope.adc.samples = int(samples)
    scope.adc.offset = int(ADC_OFFSET)
    scope.adc.basic_mode = "rising_edge"
    try:
        scope.trigger.triggers = "tio4"
    except Exception as e:
        print("Trigger config note:", e)

    if hasattr(scope.gain, "db"):
        scope.gain.db = float(gain_db)
    elif hasattr(scope.gain, "gain"):
        scope.gain.gain = int(gain_db)

def reset_target():
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high_z'
    time.sleep(0.05)
    target.flush()
    time.sleep(0.02)
    target.flush()

## 6. Python reference for `gf_mul(a, b)`

In [ ]:
# HQC GF(2^8) reference polynomial: x^8 + x^4 + x^3 + x^2 + 1 = 0x11D
GF_POLY = 0x11D

def gf_mul_ref_python(a: int, b: int) -> int:
    a &= 0xFF
    b &= 0xFF
    x = 0
    aa = a
    bb = b
    for _ in range(8):
        if bb & 1:
            x ^= aa
        bb >>= 1
        aa <<= 1
        if aa & 0x100:
            aa ^= GF_POLY
        aa &= 0xFF
    return x & 0xFF

def hw8(x: int) -> int:
    return int(bin(x & 0xFF).count("1"))

## 7. SimpleSerial v2.1 helpers

현재 펌웨어 프로토콜은 다음과 같음.

- `j`: 입력 `(a, b)` 설정, payload 2바이트, ACK만 반환
- `c`: `gf_mul(a, b)` 실행, payload 없음, ACK만 반환
- `r`: 마지막 결과 2바이트 반환

즉 **반드시 `j -> c -> r` 순서**를 지켜야 함.


In [ ]:
def ss_set_input_ab(a: int, b: int, timeout=500):
    a &= 0xFF
    b &= 0xFF
    target.simpleserial_write('j', bytes([a, b]))
    ack = target.simpleserial_wait_ack(timeout=timeout)
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ACK for 'j' failed: {ack}")
    return ack

def ss_capture_one_trace(timeout=500):
    scope.arm()
    target.simpleserial_write('c', b'')
    timed_out = scope.capture()
    if timed_out:
        raise TimeoutError("scope.capture() timed out")
    trace = np.array(scope.get_last_trace())
    ack = target.simpleserial_wait_ack(timeout=timeout)
    if ack is None or len(ack) == 0 or ack[0] != 0x00:
        raise RuntimeError(f"ACK for 'c' failed: {ack}")
    return trace

def ss_read_result(timeout=500) -> int:
    target.simpleserial_write('r', b'')
    resp = target.simpleserial_read('r', 2, timeout=timeout)
    if resp is None or len(resp) != 2:
        raise RuntimeError(f"Read for 'r' failed: {resp}")
    v_fw = (resp[0] << 8) | resp[1]   # firmware returns [MSB, LSB]
    return v_fw

def ss_set_capture_read(a: int, b: int, timeout=500, do_reset=False):
    if do_reset:
        reset_target()

    ss_set_input_ab(a, b, timeout=timeout)
    tr = ss_capture_one_trace(timeout=timeout)
    v_fw = ss_read_result(timeout=timeout)
    return tr, v_fw

## 8. Auto gain sweep

In [ ]:
def is_clipped(trace: np.ndarray, near=0.49, frac=0.002) -> bool:
    m = float(np.max(np.abs(trace)))
    if m < near:
        return False
    thr = m - 1e-4
    ratio = float(np.mean(np.abs(trace) >= thr))
    return ratio > frac

def gain_sweep(gains_db, a=0x12, b=0x34):
    rows = []
    for g in tqdm(gains_db, desc="Gain sweep"):
        configure_scope(SAMPLES, g)
        reset_target()
        tr, v_fw = ss_set_capture_read(a, b)
        rows.append({
            "gain_db": g,
            "max_abs": float(np.max(np.abs(tr))),
            "std": float(np.std(tr)),
            "clipped": bool(is_clipped(tr)),
            "v_fw": int(v_fw),
        })
    ok = [r for r in rows if not r["clipped"]]
    best = max(ok, key=lambda r: r["std"]) if ok else None
    return rows, best

GAIN_CANDIDATES = list(range(10, 41, 2))
rows, best = gain_sweep(GAIN_CANDIDATES)
print("Best:", best)
for r in rows:
    print(r)

if best:
    GAIN_DB = best["gain_db"]
    configure_scope(SAMPLES, GAIN_DB)
    reset_target()
    tr, _ = ss_set_capture_read(0x12, 0x34)
    plt.figure(figsize=(10, 3))
    plt.plot(tr)
    plt.title(f"Trace at best gain={GAIN_DB} dB")
    plt.show()

## 9. Manual gain override

In [ ]:
GAIN_DB = 40

## 10. Apply configuration

In [ ]:
configure_scope(SAMPLES, GAIN_DB)
reset_target()
print("Configured. Gain:", GAIN_DB, "Samples:", SAMPLES)

## 11. Single trace sanity check

In [ ]:
test_a = 0x12
test_b = 0x34

tr, v_fw = ss_set_capture_read(test_a, test_b)
v_py = gf_mul_ref_python(test_a, test_b)

plt.figure(figsize=(10, 3))
plt.plot(tr)
plt.title(f"1-trace (a=0x{test_a:02X}, b=0x{test_b:02X}, gain={GAIN_DB}dB)")
plt.xlabel("Sample")
plt.ylabel("ADC")
plt.show()

print(f"Firmware result = 0x{v_fw:04X}")
print(f"Python   result = 0x{v_py:02X}")
assert (v_fw & 0xFF) == v_py, "Firmware/Python gf_mul mismatch"
print("OK: one trace captured and result matched.")

## 12. Preview capture on random pairs

In [ ]:
rng = np.random.default_rng(RNG_SEED)

preview_traces = []
preview_labels = []

for _ in tqdm(range(N_TRACES_PREVIEW), desc="Preview capture"):
    a = int(rng.integers(0, 256))
    b = int(rng.integers(0, 256))
    tr, v_fw = ss_set_capture_read(a, b)
    v = v_fw & 0xFF
    preview_traces.append(np.array(tr, dtype=np.float32))
    preview_labels.append((a, b, v))

preview_traces = np.stack(preview_traces)
preview_labels = np.array(preview_labels, dtype=np.uint16)

print("preview_traces.shape =", preview_traces.shape)
print("preview_labels.shape =", preview_labels.shape)

plt.figure(figsize=(10, 3))
plt.plot(preview_traces[0])
plt.title("Preview trace[0]")
plt.show()

### 12-1. Low-level SS handshake debug (선택)

In [ ]:
reset_target()
target.flush()
time.sleep(0.1)

target.simpleserial_write('j', bytes([0x12, 0x34]))
print("ack j =", target.simpleserial_wait_ack(timeout=1000))

scope.arm()
target.simpleserial_write('c', b'')
ret = scope.capture()
print("capture ret =", ret)

ack_c = target.simpleserial_wait_ack(timeout=1000)
print("ack c =", ack_c)

target.simpleserial_write('r', b'')
resp = target.simpleserial_read('r', 2, timeout=1000)
print("resp =", resp)

## 13. Batch capture + HDF5 save

In [ ]:
def capture_to_hdf5(out_path: str, n_traces: int, chunk_size: int = 1000, seed: int = 1234):
    rng = np.random.default_rng(seed)

    with h5py.File(out_path, "w") as f:
        # metadata
        f.attrs["platform"] = PLATFORM
        f.attrs["ss_ver"] = SS_VER
        f.attrs["samples"] = int(SAMPLES)
        f.attrs["gain_db"] = float(GAIN_DB)
        f.attrs["seed"] = int(seed)

        traces_ds = f.create_dataset(
            "traces",
            shape=(n_traces, SAMPLES),
            dtype=np.float32,
            chunks=(min(chunk_size, n_traces), SAMPLES),
            compression="gzip",
        )

        a_ds = f.create_dataset("a", shape=(n_traces,), dtype=np.uint8)
        b_ds = f.create_dataset("b", shape=(n_traces,), dtype=np.uint8)
        v_ds = f.create_dataset("v", shape=(n_traces,), dtype=np.uint8)
        hw_a_ds = f.create_dataset("hw_a", shape=(n_traces,), dtype=np.uint8)
        hw_b_ds = f.create_dataset("hw_b", shape=(n_traces,), dtype=np.uint8)
        hw_v_ds = f.create_dataset("hw_v", shape=(n_traces,), dtype=np.uint8)

        idx = 0
        pbar = tqdm(total=n_traces, desc="Capturing traces")
        while idx < n_traces:
            bs = min(chunk_size, n_traces - idx)

            tr_batch = np.empty((bs, SAMPLES), dtype=np.float32)
            a_batch = np.empty((bs,), dtype=np.uint8)
            b_batch = np.empty((bs,), dtype=np.uint8)
            v_batch = np.empty((bs,), dtype=np.uint8)
            hw_a_batch = np.empty((bs,), dtype=np.uint8)
            hw_b_batch = np.empty((bs,), dtype=np.uint8)
            hw_v_batch = np.empty((bs,), dtype=np.uint8)

            for i in range(bs):
                a = int(rng.integers(0, 256))
                b = int(rng.integers(0, 256))
                tr, v_fw = ss_set_capture_read(a, b)
                v = v_fw & 0xFF

                tr_batch[i] = np.array(tr, dtype=np.float32)
                a_batch[i] = a
                b_batch[i] = b
                v_batch[i] = v
                hw_a_batch[i] = hw8(a)
                hw_b_batch[i] = hw8(b)
                hw_v_batch[i] = hw8(v)

            traces_ds[idx:idx+bs] = tr_batch
            a_ds[idx:idx+bs] = a_batch
            b_ds[idx:idx+bs] = b_batch
            v_ds[idx:idx+bs] = v_batch
            hw_a_ds[idx:idx+bs] = hw_a_batch
            hw_b_ds[idx:idx+bs] = hw_b_batch
            hw_v_ds[idx:idx+bs] = hw_v_batch

            idx += bs
            pbar.update(bs)

        pbar.close()

    print("Saved:", out_path)

# 먼저 소규모로 확인한 뒤 늘릴 것
capture_to_hdf5(OUT_PATH, n_traces=N_TRACES_CAPTURE, chunk_size=1000, seed=RNG_SEED)

## 14. Quick inspection of saved file

In [ ]:
with h5py.File(OUT_PATH, "r") as f:
    print("keys:", list(f.keys()))
    print("traces shape:", f["traces"].shape)
    print("example a,b,v:", int(f["a"][0]), int(f["b"][0]), int(f["v"][0]))

    plt.figure(figsize=(10, 3))
    plt.plot(f["traces"][0])
    plt.title("Saved trace[0]")
    plt.show()

---

**Next:**
- `gf_mul_lra.ipynb` — 저장된 H5 파일로 LRA leakage assessment
- `gf_mul_lda.ipynb` — 저장된 H5 파일로 LDA template attack (6 templates, Table 2)
